# Model Hub - Using Hugging Face libraries

This notebook will go through using Hugging Face libraries with models stored in H2O Model Hub.

We will cover:
- Determining H2O AI Cloud values to use with Hugging Face libraries
  - Endpoint URL
  - Access token
- Using Hugging Face libraries with models in H2O AI Cloud
  - Configuring the Hugging Face library
  - Example: Using the `pipeline` API from the Hugging Face `transformers` library
  - Example: Downloading single model files with the Hugging Face `huggingface_hub` library

## Setup

Let's install packages we'll use, and decide on the model we'll be playing with.

In [ ]:
import sys

!{sys.executable} -m pip install -q "h2o-authn[discovery]"
!{sys.executable} -m pip install -q huggingface_hub
!{sys.executable} -m pip install -q transformers
!{sys.executable} -m pip install -q torch --index-url https://download.pytorch.org/whl/cpu

> 📢 Important
>
> This notebook makes the assumption that a model has already been uploaded into H2O Model Hub. This is not the case for new environments. Replace the value below with a model which you know exists in your environment.

Let's decide on the model that we'll be reading from Model Hub. The model is in standard Hugging Face "repo ID" format.

In [ ]:
available_modelhub_model = "albert/albert-base-v2"

## Determining H2O AI Cloud values to use with Hugging Face libraries

> 📢 Important
> 
> This section assumes that an H2O AI Cloud environment can be discovered from your environment.
> On local environments, this means having the H2O CLI installed and configured.
>
> For information on other ways to discover required H2O AI Cloud, please see the notebook titled _"Drive - Connecting from different environments"_.

We'll start by discoverying the H2O AI Cloud the environment has been configured for.

In [ ]:
import h2o_discovery

discovery = h2o_discovery.discover()

### Endpoint URL

We can use the `discovery` object to find the URL for H2O Model Hub, the H2O service serving a set of Hugging Face compatible APIs.

In [ ]:
base_modelhub_url = discovery.services["modelhub"].uri

A neat feature of Model Hub is it's support for multiple, isolated, virtual registries.

For this notebook tutorial, we're only concerned with the "global" registry. By default,the "global" registry grants all authenticated users read access by default. This is analogous to Hugging Face allowing users to download any public model.

The final endpoint is a combination of Model Hub's base URL and the virtual registry ("global") we want to work with.

In [ ]:
modelhub_endpoint = f"{base_modelhub_url}/global"

### Access token

All requests to Model Hub must be authenticated. The `h2o_authn` package supplies a helper for creating a token provider for H2O AI Cloud.

We create that token provider, and use it generate an access token allowing us access to Model Hub.

In [ ]:
import h2o_authn.discovery

token_provider = h2o_authn.discovery.create(discovery)
access_token = str(token_provider.token())

## Using Hugging Face libraries with models in H2O AI Cloud

### Configuring the Hugging Face library

For Hugging Face Python libraries to route requests to H2O's Model Hub, we set the `HF_ENDPOINT` environment variable. **This must be set before any Hugging Face library is loaded for the first time.**

Some Hugging Face library functions take an `endpoint` argument, which could be set in place of setting an environment variable.

In [ ]:
import os

os.environ["HF_ENDPOINT"] = modelhub_endpoint

Setting the `HF_TOKEN` environment variable with our H2O access token will allow Hugging Face library operations to pass H2O AI Cloud authentication.

Some Hugging Face library functions take a `token` argument, which could be set in place of setting an environment variable.

In [ ]:
os.environ["HF_TOKEN"] = access_token

### Example: Using the `pipeline` API from the Hugging Face `transformers` library

Let's go through a demonstration of serving models in H2O AI Cloud via Hugging Face libraries.

The `pipeline` API is a popular API in Hugging Face's `transformers` library. Pipelines group together a pretrained model with the preprocessing that was used during that model's training.

Having already set `HF_ENDPOINT` and `HF_TOKEN` to point to, and authenticate with, H2O AI Cloud, we use the Hugging Face `pipelines` API as per usual.

In [ ]:
from transformers import pipeline

sentiment_analysis = pipeline(
    "sentiment-analysis",
    model=available_modelhub_model,
)

In [ ]:
sentiment_analysis("That's a pretty cool shirt.")

### Example: Downloading single model files with the Hugging Face `huggingface_hub` library

A common way to download individual model files is via Hugging Face's `hf_hub_download` function. It downloads the remote file, caches it on disk (in a version-aware way), and returns its local file path.

Here, we use it to download and cache the `config.json` file from our model in H2O AI Cloud.

In [ ]:
import huggingface_hub

huggingface_hub.hf_hub_download(repo_id=available_modelhub_model, filename="config.json")